## 1. Definitions

In [1]:
from pathlib import Path
import os
import shutil
import json
import re
import pandas as pd
import subprocess


#---Data---#
BMS = ["nodeapp","mediawiki","compression","dacapo-spring","dacapo-luindex","dacapo-lusearch","renaissance-http","renaissance-chirper",
       "502.gcc_r.gcc-pp.opts-O3_-finline-limit_36000","505.mcf_r.inp","523.xalancbmk_r.xalanc","531.deepsjeng_r.ref","541.leela_r.ref"]

CONF = [2,4,6,8,12,16,24,32,40,50,60,75,100,125,150,175,200]

STATS_PREFIX = {
    "multicore": "board.processor.cores1.core.",
    "singlecore": "board.processor.cores.core."
}

PDFS = {
    "penalty_lin": "valuePred.penaltyCycles::",
    "savings_lin": "valuePred.valuePredSavedCycles::",
    "penalty_log": "valuePred.penaltyCyclesLog2::",
    "savings_log": "valuePred.valuePredSavedCyclesLog2::",
}

STATS = {
    "ipc": "ipc",
    "coverage": "valuePred.predCoverage",
    "accuracy": "valuePred.accuracy",
    "penalty_total": "valuePred.totalPenaltyCycles",
    "savings_total": "valuePred.totalSavedCycles",
    "pdfs": PDFS
}

data = {
    "script_file": "../scripts/run_important_simpoints.sh",
    "config_file": "../gem5-configs/all-simpoint-run.py",
    "configs_path": "../gem5-configs/conf-configs",
    "stats_file" : "stats.txt",
    "min-confidence" : 2,
    "max-confidence" : 31,
    "confidence" : CONF, 
    "bms" : BMS,
    "experiment" : "noStride_conf127_delay8_rtz_64KiBTab",
    "stats" : STATS,
    "stats_prefix" : STATS_PREFIX,
    "use_stride" : False
}

data["result_path"]= f"results/{data["experiment"]}/"
data["stats_path"] = f"../scripts/results/arm64/{data["experiment"]}/"

with open("config.json", "w") as f:
    json.dump(data, f, indent=4)


#---CHECKS---#
if data["min-confidence"]>=data["max-confidence"]:
    raise ValueError("Check Confidence values")


#---METHODS----#

def contains_line(path,target_line):
    lines = Path(path).read_text().splitlines()
    if target_line in lines:
        return True
    else:
        return False
    
def contains_regex(path,target_regex):
    lines = Path(path).read_text().splitlines()
    return any(re.search(target_regex,line) for line in lines)

def replace_line(path,target_string,new_line):
    lines = Path(path).read_text().splitlines()
    updated = False
    for i, line in enumerate(lines):
        if line.strip().startswith(target_string):
            lines[i] = new_line
            updated = True
            break
    if not updated:
        raise ValueError(f"Key '{target_string}' not found in {path}")
    Path(path).write_text("\n".join(lines) + "\n")


def insert_line_after(path,target_line,new_line):
    lines = Path(path).read_text().splitlines()
    updated = False
    for i, line in enumerate(lines):
        if line.strip().startswith(target_line):
            lines.insert(i+1,new_line)
            updated = True
            break
    if not updated:
        raise ValueError(f"Key '{target_line}' not found in {path}")
    Path(path).write_text("\n".join(lines) + "\n")

def remove_line(path,target_line):
    lines = Path(path).read_text().splitlines()
    updated = False
    for i, line in enumerate(lines):
        if line.strip().startswith(target_line):
            lines.remove(line)
            updated = True
            break
    if not updated:
        print(f"Key '{target_line}' not found in {path}")
    Path(path).write_text("\n".join(lines) + "\n")




## 2 Collecting Stats
#### Functions

In [2]:
def read_stats(prefix, stats, file_path, n):
    result = {}

    counters = {name: 0 for name in stats}
    with open(file_path, "r") as f:
        for line in f:
            for name in stats:
                if name != "pdfs":
                    if (prefix+stats[name]) in line:
                        counters[name] += 1
                    if counters[name] == n:
                        result[name] = float(line.split()[1])
                        counters[name] = n+1
    

    # Evaluate pdfs for penalty and savings
    for pdf in stats["pdfs"]:
        result[pdf]=read_pdf(prefix, pdf, stats, file_path, n)

    missing=set(stats)-set(result)
    if len(missing) > 1:
        raise ValueError(f"{missing} not found")
    return result

def read_pdf(prefix, pdf, stats, file_path, n):
    result = {}
    if "log" in pdf:
        for i in range(0,16,1):
            stat_name = prefix+stats["pdfs"][pdf]+f"{i} "
            counter = 0
            with open(file_path, "r") as f:
                for line in f:
                    if stat_name in line:
                        counter += 1
                    if counter == n:
                        result[2**i] = float(line.split()[1])
                        break
    elif "lin" in pdf:
        for i in range(0,42,2):
            if i < 40:
                stat_name = prefix+stats["pdfs"][pdf]+f"{i}-"
                i=i+1
            else:
                stat_name = prefix+stats["pdfs"][pdf]+f"{i}"
            counter = 0
            with open(file_path, "r") as f:
                for line in f:
                    if stat_name in line:
                        counter += 1
                    if counter == n:
                        result[i] = float(line.split()[1])
                        break
    else: 
        raise ValueError(f"{stats} pdf not defined")
    
    stat_name = prefix+stats["pdfs"][pdf]+"total"
    counter = 0
    with open(file_path, "r") as f:
        for line in f:
            if stat_name in line:
                counter += 1
            if counter == n:
                result["total"] = float(line.split()[1])
                break


    return result


### 2.1 Different Confidence Values

In [4]:




#----------
# Set Experiment to specify Stats and Result Path to experiment directory
# Which stat dump to use:
n=2
data["experiment"]=data["experiment"]="noStride_newConf_delay8_rtz_64KiBTab"
print(f"Collecting data from experiment{data["experiment"]}")

data["stats_path"] = f"../scripts/results/arm64/{data["experiment"]}/"
data["result_path"]=f"results/{data["experiment"]}/"
#----------

Path(data["result_path"]).mkdir(exist_ok=True)



for bm in data["bms"]:
    results=[]

    file_path=f"{data["stats_path"]}{bm}/{min(data["confidence"])}/{data["stats_file"]}"
    print("Processing "+file_path+" etc.")
    multicore= True
    if contains_regex(file_path,data["stats_prefix"]["singlecore"]+data["stats"]["ipc"]):
        multicore = False
    for confidence in data["confidence"]:
        file_path=f"{data["stats_path"]}{bm}/{confidence}/{data["stats_file"]}"
        if multicore:
            metrics_1 = read_stats(data["stats_prefix"]["multicore"],data["stats"],file_path, n)            
            results.append({
                "confidence": confidence,
                **metrics_1
            })
        else:
            metrics = read_stats(data["stats_prefix"]["singlecore"],data["stats"],file_path, n)
            results.append({
                "confidence": confidence,
                **metrics
            })
    df=pd.DataFrame(results)
    df.to_csv(f"{data["result_path"]}results_{bm}.csv",index=False)


Processing ../scripts/results/arm64/noStride_newConf_delay8_rtz_64KiBTab/nodeapp/2/stats.txt etc.
Processing ../scripts/results/arm64/noStride_newConf_delay8_rtz_64KiBTab/mediawiki/2/stats.txt etc.
Processing ../scripts/results/arm64/noStride_newConf_delay8_rtz_64KiBTab/compression/2/stats.txt etc.
Processing ../scripts/results/arm64/noStride_newConf_delay8_rtz_64KiBTab/dacapo-spring/2/stats.txt etc.
Processing ../scripts/results/arm64/noStride_newConf_delay8_rtz_64KiBTab/dacapo-luindex/2/stats.txt etc.
Processing ../scripts/results/arm64/noStride_newConf_delay8_rtz_64KiBTab/dacapo-lusearch/2/stats.txt etc.
Processing ../scripts/results/arm64/noStride_newConf_delay8_rtz_64KiBTab/renaissance-http/2/stats.txt etc.
Processing ../scripts/results/arm64/noStride_newConf_delay8_rtz_64KiBTab/renaissance-chirper/2/stats.txt etc.
Processing ../scripts/results/arm64/noStride_newConf_delay8_rtz_64KiBTab/502.gcc_r.gcc-pp.opts-O3_-finline-limit_36000/2/stats.txt etc.
Processing ../scripts/results/ar

### 2.2 Baselinestats / One Confidence value

In [14]:

#----------
# Set Experiment to specify Stats and Result Path to experiment directory
# Which stat dump to use:
n=2
data["experiment"]=data["experiment"]="baseCase_delay8"
print(f"Collecting data from experiment {data["experiment"]}")

data["stats_path"] = f"../scripts/results/arm64/{data["experiment"]}/"
data["result_path"]=f"results/{data["experiment"]}/"
#----------

Path(data["result_path"]).mkdir(exist_ok=True)

with open(data["stats_path"]+"config.json", "r") as f:
    temp = json.load(f)
    conf = temp["min-confidence"]

results=[]
for bm in data["bms"]:

    file_path=f"{data["stats_path"]}{bm}/{data["stats_file"]}"
    print("Processing "+file_path)
    multicore= True
    if contains_regex(file_path,data["stats_prefix"]["singlecore"]+data["stats"]["ipc"]):
        metrics = read_stats(data["stats_prefix"]["singlecore"],data["stats"],file_path, n)            
    else:
        metrics = read_stats(data["stats_prefix"]["multicore"],data["stats"],file_path, n)
    print(metrics)
    results.append({
        "confidence": conf,
        "benchmark": bm,
        **metrics
    })
df=pd.DataFrame(results)
df.to_csv(f"{data["result_path"]}results.csv",index=False)

Processing ../scripts/results/arm64/baseCase_delay8/nodeapp/stats.txt
{'ipc': 1.200538, 'coverage': nan, 'accuracy': nan, 'penalty_total': 0.0, 'savings_total': 0.0, 'penalty_lin': {1: 0.0, 3: 0.0, 5: 0.0, 7: 0.0, 9: 0.0, 11: 0.0, 13: 0.0, 15: 0.0, 17: 0.0, 19: 0.0, 21: 0.0, 23: 0.0, 25: 0.0, 27: 0.0, 29: 0.0, 31: 0.0, 33: 0.0, 35: 0.0, 37: 0.0, 39: 0.0, 40: 0.0, 'total': 0.0}, 'savings_lin': {1: 0.0, 3: 0.0, 5: 0.0, 7: 0.0, 9: 0.0, 11: 0.0, 13: 0.0, 15: 0.0, 17: 0.0, 19: 0.0, 21: 0.0, 23: 0.0, 25: 0.0, 27: 0.0, 29: 0.0, 31: 0.0, 33: 0.0, 35: 0.0, 37: 0.0, 39: 0.0, 40: 0.0, 'total': 0.0}, 'penalty_log': {1: 0.0, 2: 0.0, 4: 0.0, 8: 0.0, 16: 0.0, 32: 0.0, 64: 0.0, 128: 0.0, 256: 0.0, 512: 0.0, 1024: 0.0, 2048: 0.0, 4096: 0.0, 8192: 0.0, 16384: 0.0, 32768: 0.0, 'total': 0.0}, 'savings_log': {1: 0.0, 2: 0.0, 4: 0.0, 8: 0.0, 16: 0.0, 32: 0.0, 64: 0.0, 128: 0.0, 256: 0.0, 512: 0.0, 1024: 0.0, 2048: 0.0, 4096: 0.0, 8192: 0.0, 16384: 0.0, 32768: 0.0, 'total': 0.0}}
Processing ../scripts/result